In [20]:
import os
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession
import pyarrow
import fastparquet
import pandas as pd

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/21/26 11:31:23] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=29495;file://c:\anaconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=916531;file://c:\anaconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


# Importar nodos

In [21]:
# 2. Importar todas las funciones del archivo nodes.py
import central_perm_flow.pipelines.data_processing.nodes as nodes

# Esto te dará una lista limpia de todo lo definido en ese archivo de nodos
print(dir(nodes))


['Tuple', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__warningregistry__', 'central_preprocessing_calaca', 'central_preprocessing_estaca', 'check_and_export_duplicates', 'check_dataframe', 'clean_column_names', 'clean_column_objects', 'clean_nulls', 'convert_all_standardized_dates', 'convert_standardized_dates', 'datetime', 'duckdb', 'np', 'numeric_conversion_node', 'pd', 're', 'remove_accents', 'remove_accents_and_special_chars', 'select_columns', 'unicodedata']


In [22]:
# Muestra la ayuda formateada
help(nodes.select_columns)

# O simplemente imprime el texto de la descripción
print(nodes.select_columns.__doc__)

Help on function select_columns in module central_perm_flow.pipelines.data_processing.nodes:

select_columns(df: pandas.DataFrame, columns: list) -> pandas.DataFrame
    Selecciona un subconjunto de columnas del DataFrame.
    Args:
        df: El DataFrame del cual seleccionar columnas.
        columns: Una lista de nombres de columnas a seleccionar.
    Returns:
        pd.DataFrame: Un nuevo DataFrame que contiene solo las columnas seleccionadas.

Selecciona un subconjunto de columnas del DataFrame.
Args:
    df: El DataFrame del cual seleccionar columnas.
    columns: Una lista de nombres de columnas a seleccionar.
Returns:
    pd.DataFrame: Un nuevo DataFrame que contiene solo las columnas seleccionadas.




# Fuentes de Información

## Base de Estados de los Estudiantes

In [ ]:
# Base de central con fechas de bajas, fechas de graduación
central_caracterizacion = catalog.load('central_caracterizacion')

### Parámetros

In [ ]:
central_col_fechas = ['fecha_creacion', 'fecha_nacimiento', 'inicio_clases']
central_col_emails = ['correo','correo_electronico_del_contacto_emergencia']
central_col_dd = ['identidad']
central_col_sort = ['identidad']

In [ ]:
# Limpieza de la base de operaciones
central_caracterizacion = nodes.clean_column_names(central_caracterizacion)
central_caracterizacion = nodes.convert_all_standardized_dates(central_caracterizacion,central_col_fechas)
central_caracterizacion['identidad'] = pd.to_numeric(central_caracterizacion['identidad'], errors='coerce')
central_caracterizacion = nodes.clean_column_objects(central_caracterizacion,central_col_emails)
central_caracterizacion_sd, central_caracterizacion_cd  = nodes.check_and_export_duplicates(central_caracterizacion, subset=central_col_dd, col_ordenar = central_col_sort)
central_caracterizacion = central_caracterizacion.drop(columns='nivel') 



## Calendario Académico

In [ ]:
# Calendario Académico
central_estados_calac = catalog.load("central_estados_calac")

### Parámetros

In [ ]:
central_caracterizacion = central_estados_calac.merge(central_caracterizacion, how='left', left_on=['identificacion'], right_on=['identidad'])

In [ ]:

# Base de central con fechas de bajas, fechas de graduación
central_caracterizacion = catalog.load('central_caracterizacion')
central_col_fechas = ['fecha_creacion', 'fecha_nacimiento', 'inicio_clases']
central_col_emails = ['correo','correo_electronico_del_contacto_emergencia']
central_col_dd = ['identidad']
central_col_sort = ['identidad']
# Limpieza de la base de operaciones
central_caracterizacion = nodes.clean_column_names(central_caracterizacion)
central_caracterizacion = nodes.convert_all_standardized_dates(central_caracterizacion,central_col_fechas)
central_caracterizacion['identidad'] = pd.to_numeric(central_caracterizacion['identidad'], errors='coerce')
central_caracterizacion = nodes.clean_column_objects(central_caracterizacion,central_col_emails)
central_caracterizacion_sd, central_caracterizacion_cd  = nodes.check_and_export_duplicates(central_caracterizacion, subset=central_col_dd, col_ordenar = central_col_sort)
central_caracterizacion = central_caracterizacion.drop(columns='nivel') 

# Calendario Académico
central_estados_calac = catalog.load("central_estados_calac")
central_caracterizacion = central_estados_calac.merge(central_caracterizacion, how='left', left_on=['identificacion'], right_on=['identidad'])





## Nodo

In [23]:
import pandas as pd

def transformar_caracterizacion_central(
    df_caracterizacion: pd.DataFrame,
    df_estados_calac: pd.DataFrame,
    params_fechas: list,
    params_emails: list,
    params_duplicados: list,
    params_orden: list,
    params_columnas_caracterizacion: list
) -> pd.DataFrame:
    """
    Función para limpiar la base de operaciones y cruzarla con el calendario académico.
    """
    
    # 1. Limpieza de nombres de columnas
    df = nodes.clean_column_names(df_caracterizacion)
    
    # 2. Conversión de fechas usando los parámetros del YML
    df = nodes.convert_all_standardized_dates(df, params_fechas)
    
    # 3. Conversión de identidad a numérico (Coerce para manejar errores)
    df['identidad'] = pd.to_numeric(df['identidad'], errors='coerce')
    
    # 4. Limpieza de correos/objetos
    df = nodes.clean_column_objects(df, params_emails)
    
    # 5. Manejo de duplicados (obtenemos la base sin duplicados 'sd')
    df_sd, _ = nodes.check_and_export_duplicates(
        df, 
        subset=params_duplicados, 
        col_ordenar=params_orden
    )
    
    # 6. Eliminar columna 'nivel' si existe
    df_sd = df_sd.drop(columns='nivel', errors='ignore')

    # 7. Seleccionar columnas especificas
    df_sd = nodes.select_columns(df_sd, params_columnas_caracterizacion)

    
    # 8. Cruce con Calendario Académico (Merge)
    # Nota: Usamos df_sd para asegurar que el cruce no duplique filas inesperadamente
    resultado = df_estados_calac.merge(
        df_sd, 
        how='left', 
        left_on=['identificacion'], 
        right_on=['identidad']
    )
    
    return resultado

In [24]:
parameters = catalog.load("parameters")
#parameters['central_caracterizacion_params']

[04/21/26 11:31:35] INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=366122;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=245101;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [25]:
# Kedro carga automáticamente la variable 'catalog'
df_caracterizacion = catalog.load("central_caracterizacion")
df_estados_calac = catalog.load("central_estados_calac")


[04/21/26 11:31:39] INFO     Loading data from central_caracterizacion (CSVDataset)...         ]8;id=638414;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=165753;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

[04/21/26 11:31:40] INFO     Loading data from central_estados_calac (ExcelDataset)...         ]8;id=352166;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=568197;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [26]:
df_estados_calac_caracterizacion = transformar_caracterizacion_central(
    df_caracterizacion=df_caracterizacion,
    df_estados_calac=df_estados_calac,
    params_fechas=parameters['central_caracterizacion_params']['col_fechas'],
    params_emails=parameters['central_caracterizacion_params']['col_emails'],
    params_duplicados=parameters['central_caracterizacion_params']['col_dd'],
    params_orden=parameters['central_caracterizacion_params']['col_sort'],
    params_columnas_caracterizacion=parameters['central_caracterizacion_params']['params_columnas_caracterizacion']
)

[04/21/26 11:31:43] WARNING  G:\Unidades compartidas\Alianzas\3.                                    ]8;id=904680;file://c:\anaconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=375106;file://c:\anaconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             Data\CENTRAL\central-perm-flow\src\central_perm_flow\pipelines\data_pr                
                             ocessing\nodes.py:105: UserWarning: Parsing dates in %d/%m/%Y format                  
                             when dayfirst=False (the default) was specified. Pass `dayfirst=True`                 
                             or specify a format to silence this warning.                                          
                               df = pd.to_datetime(df, errors='coerce')                                            
                                                                                                                   

                    WARNING  G:\Unidades compartidas\Alianzas\3.                                    ]8;id=963972;file://c:\anaconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=584473;file://c:\anaconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             Data\CENTRAL\central-perm-flow\src\central_perm_flow\pipelines\data_pr                
                             ocessing\nodes.py:105: UserWarning: Parsing dates in %d/%m/%Y format                  
                             when dayfirst=False (the default) was specified. Pass `dayfirst=True`                 
                             or specify a format to silence this warning.                                          
                               df = pd.to_datetime(df, errors='coerce')                                            
                                                                                                                   

In [ ]:
from kedro.pipeline import Pipeline, node, pipeline
from .nodes import transformar_caracterizacion_central

def create_pipeline(**kwargs) -> Pipeline:
    return pipeline(
        [
            node(
                func=transformar_caracterizacion_central,
                inputs=[
                    "central_caracterizacion",      # Dataset del catalog.yml
                    "central_estados_calac",       # Dataset del catalog.yml
                    "params:central_processing.col_fechas",
                    "params:central_processing.col_emails",
                    "params:central_processing.col_dd",
                    "params:central_processing.col_sort",
                    "params:central_processing.params_columnas_caracterizacion"
                ],
                outputs="central_caracterizacion_final", # Nombre del resultado
                name="nodo_transformar_central",
            ),
        ]
    )

## EDA

In [27]:
df_estados_calac_caracterizacion.columns


Index(['identificacion', 'codigo_sis', 'id_inconcert', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'cohorte_inicial',
       'cohorte_actual', 'nivel', 'nivel_academico', 'programa', 'estado',
       'bloque', 'descuentos', 'fecha_ingreso', 'fecha_de_registro',
       'fecha_de_baja_t', 'fecha_de_baja_d', 'fecha_baja',
       'fecha_de_reingreso', 'fecha_grado', 'fecha_activo', 'tipo_baja',
       'motivo_baja', 'submotivo_baja', 'comentarios', 'periodo_raw',
       'fecha_inicio', 'fecha_fin', 'shifted_fecha_inicio', 'semana',
       'semana_acumulada', 'month', 'mes_academico', 'anio_gregoriano',
       'mes_gregoriano', 'student_journey', 'tipo', 'max_semana_teorica',
       'exito_estudiantil', 'etapa_studen_journey', 'ai', 'di', 'gi', 'engi',
       'ci', 'matricula', 'id_alumno', 'tipo_identidad', 'identidad',
       'primer_nombre', 'segundo_nombre', 'apellido_paterno',
       'apellido_materno', 'genero', 'fecha_nacimiento', 'correo',
       'telefono_movi

In [28]:
df_estados_calac_caracterizacion.loc[:,['genero','estado']].value_counts()


genero     estado              
femenino   activo                  742
masculino  activo                  680
femenino   egresado no graduado    105
           baja definitiva          77
masculino  baja definitiva          61
           egresado no graduado     40
           baja temporal            24
femenino   baja temporal            24
Name: count, dtype: int64

In [30]:
df_estados_calac_caracterizacion.loc[:,['genero','estrato_social']].value_counts()


genero     estrato_social
femenino   tres              481
masculino  tres              411
femenino   dos               274
masculino  dos               220
femenino   cuatro            140
masculino  cuatro            122
           uno                29
femenino   uno                27
           cinco              21
masculino  cinco              17
           no aplica           4
femenino   seis                3
masculino  seis                2
femenino   no aplica           2
Name: count, dtype: int64

In [31]:
df_estados_calac_caracterizacion.loc[:,['estado','estrato_social']].value_counts()


estado                estrato_social
activo                tres              724
                      dos               415
                      cuatro            200
baja definitiva       tres               76
egresado no graduado  tres               68
                      dos                43
activo                uno                42
                      cinco              31
baja definitiva       dos                27
egresado no graduado  cuatro             26
baja temporal         tres               24
baja definitiva       cuatro             22
baja temporal         cuatro             14
                      dos                 9
baja definitiva       uno                 7
egresado no graduado  uno                 6
activo                no aplica           6
baja definitiva       cinco               5
activo                seis                4
egresado no graduado  cinco               2
baja temporal         uno                 1
baja definitiva       seis            